# Experiment [1] Visual Dashboard: Temporal JEPA v0

First full CropHarvest JEPA run. Use this notebook to inspect whether frozen JEPA embeddings beat raw flattened features across label budgets and stress conditions.

In [1]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts" / "[1]"
DATA = ROOT / "data"

PALETTE = {
    "surf_jepa_v0": "#2563eb",
    "raw_flattened": "#64748b",
    "raw_stats": "#f97316",
    "surf_jepa_v0_plus_raw_stats": "#16a34a",
    "large_dual_s2": "#2563eb",
    "large_default": "#7c3aed",
    "large_dual_s2_jepa": "#2563eb",
    "large_default_jepa": "#7c3aed",
    "presto": "#db2777",
    "olmoearth": "#059669",
}
PRIORITY_CONDITIONS = ["clean", "sensor_off_s2", "temporal_drop_50", "temporal_drop_70", "s2_off_tdrop50"]
HOLDOUT_ORDER = ["rwanda-ceo", "togo", "togo-eval", "ethiopia", "lem-brazil"]

pd.options.display.max_columns = 80
pd.options.display.max_rows = 80


def standardize_table(df):
    if df.empty:
        return df
    rename = {
        "balanced_acc": "balanced_accuracy",
        "bal_acc": "balanced_accuracy",
        "mean_auc": "auc",
        "mean_f1": "f1",
        "budget": "label_budget",
    }
    return df.rename(columns={k: v for k, v in rename.items() if k in df.columns and v not in df.columns})


def read_csv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"missing: {path.relative_to(ROOT) if path.is_absolute() and ROOT in path.parents else path}")
        return pd.DataFrame()
    return standardize_table(pd.read_csv(path, **kwargs))


def read_json(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text())


def pct(x):
    return f"{100*x:.1f}%" if pd.notna(x) else ""


def scorecard(df, cols, title="Scorecard", by=None):
    if df.empty:
        print("No data for scorecard")
        return
    work = df.copy()
    if by:
        work = work.set_index(by)
    display(work[cols].style.format({c: "{:.4f}" for c in cols if pd.api.types.is_numeric_dtype(work[c])}).set_caption(title))


def bar_metric(df, x, y, color=None, title=None, barmode="group", text=True, height=460):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.bar(
        df,
        x=x,
        y=y,
        color=color,
        barmode=barmode,
        text_auto=".3f" if text else False,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="", xaxis_title="", yaxis_title=y)
    fig.show()
    return fig


def line_metric(df, x, y, color=None, facet_col=None, title=None, markers=True, height=520):
    if df.empty:
        print("No data to plot")
        return None
    fig = px.line(
        df,
        x=x,
        y=y,
        color=color,
        facet_col=facet_col,
        markers=markers,
        color_discrete_map=PALETTE,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", legend_title_text="")
    fig.show()
    return fig


def heatmap(df, x, y, z, title=None, height=520, color_continuous_scale="RdYlGn"):
    if df.empty:
        print("No data to plot")
        return None
    pivot = df.pivot_table(index=y, columns=x, values=z, aggfunc="mean")
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale=color_continuous_scale,
        height=height,
        title=title,
    )
    fig.update_layout(template="plotly_white", xaxis_title=x, yaxis_title=y)
    fig.show()
    return fig


def priority_average(df, group_cols, metric_cols=("f1", "auc", "balanced_accuracy"), conditions=PRIORITY_CONDITIONS):
    if df.empty:
        return df
    work = df[df["condition"].isin(conditions)].copy() if "condition" in df else df.copy()
    work = work.rename(columns={"balanced_acc": "balanced_accuracy", "mean_auc": "auc", "mean_f1": "f1"})
    available = [c for c in metric_cols if c in work.columns]
    if not available:
        return work.groupby(group_cols, dropna=False).size().reset_index(name="n")
    return work.groupby(group_cols, dropna=False)[available].mean().reset_index()


def normalize_model_name(name):
    text = str(name)
    return text.replace("_seed42", "").replace("_seed7", "").replace("_seed11", "")

RUNS = [ARTIFACTS / "cropharvest_jepa_v0_seed42", ARTIFACTS / "cropharvest_jepa_v0_seed7"]
runs = [p for p in RUNS if p.exists()]
print("runs:", [p.name for p in runs])

runs: ['cropharvest_jepa_v0_seed42', 'cropharvest_jepa_v0_seed7']


In [2]:
# Training curves
histories = []
for run in runs:
    hist = pd.DataFrame(read_json(run / "train_history.json"))
    if not hist.empty:
        hist["run"] = run.name
        histories.append(hist)
history = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()
if not history.empty:
    display(history.groupby("run").tail(1))
    for metric in ["train_loss", "val_loss"]:
        if metric in history:
            line_metric(history, x="epoch", y=metric, color="run", title=f"[1] {metric}")

,best_val_loss,train_seconds,history,run
99,0.018147,1816.814942,"{'epoch': 100, 'train_loss': 0.194859671371954...",cropharvest_jepa_v0_seed42
199,0.032307,1812.590104,"{'epoch': 100, 'train_loss': 0.183982749780019...",cropharvest_jepa_v0_seed7


In [3]:
# Probe results across seeds
frames = []
for run in runs:
    df = read_csv(run / "probe_results.csv")
    if not df.empty:
        df["run"] = run.name
        df["seed"] = run.name.split("seed")[-1]
        frames.append(df)
probe = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
probe.head()

,model,condition,label_budget,n_train_sub,n_test,f1,auc,balanced_accuracy,run,seed
0,surf_jepa_v0,clean,0.01,541,6770,0.768927,0.784061,0.710461,cropharvest_jepa_v0_seed42,42
1,surf_jepa_v0,clean,0.05,2707,6770,0.786632,0.821859,0.743038,cropharvest_jepa_v0_seed42,42
2,surf_jepa_v0,clean,0.10,5415,6770,0.789882,0.830310,0.754121,cropharvest_jepa_v0_seed42,42
3,surf_jepa_v0,clean,0.25,13538,6770,0.794386,0.835537,0.755538,cropharvest_jepa_v0_seed42,42
4,surf_jepa_v0,clean,1.00,54152,6770,0.797615,0.845614,0.760070,cropharvest_jepa_v0_seed42,42


In [4]:
# Main scorecard: priority conditions averaged over budgets and seeds.
priority = priority_average(probe, ["model"])
scorecard(priority, ["f1", "auc", "balanced_accuracy"], "Priority-condition mean", by="model")
bar_metric(priority.melt(id_vars="model", value_vars=["f1", "auc", "balanced_accuracy"], var_name="metric", value_name="value"), x="metric", y="value", color="model", title="[1] JEPA vs raw: priority average")

,f1,auc,balanced_accuracy
model,,,
raw_flattened,0.7579,0.7547,0.6894
surf_jepa_v0,0.7718,0.7874,0.7148


In [5]:
# Stress condition view.
cond = probe.groupby(["model", "condition"])[["f1", "auc", "balanced_accuracy"]].mean().reset_index()
heatmap(cond, x="condition", y="model", z="f1", title="[1] F1 by stress condition")
heatmap(cond, x="condition", y="model", z="auc", title="[1] AUROC by stress condition", color_continuous_scale="Viridis")

In [6]:
# Label-efficiency curves.
focus_conditions = ["clean", "sensor_off_s2", "temporal_drop_50", "temporal_drop_70", "s2_off_tdrop50"]
curve = probe[probe["condition"].isin(focus_conditions)].groupby(["model", "condition", "label_budget"])[["f1", "auc"]].mean().reset_index()
line_metric(curve, x="label_budget", y="f1", color="model", facet_col="condition", title="[1] F1 over label budget", height=620)
line_metric(curve, x="label_budget", y="auc", color="model", facet_col="condition", title="[1] AUROC over label budget", height=620)

In [7]:
# Robustness delta against clean.
cond_piv = cond.pivot(index="model", columns="condition", values="f1")
delta = cond_piv.sub(cond_piv["clean"], axis=0).reset_index().melt(id_vars="model", var_name="condition", value_name="f1_delta_vs_clean")
bar_metric(delta[delta["condition"].ne("clean")], x="condition", y="f1_delta_vs_clean", color="model", title="[1] F1 degradation vs clean", text=False, height=520)